# 34장 · LangGraph의 선택적 도구 호출과 반복

검색을 도구로 둔 그래프는 질문에 따라 검색을 건너뛰거나 다시 요청할 수 있습니다. 고정 검색 경로와 실행 횟수가 달라지는 이유를 살펴봅니다.

같은 질문을 필수 검색 노드와 선택형 검색 도구에 보냅니다. 검색 횟수와 반환 문서를 함께 기록해 ‘검색 생략’과 ‘검색했으나 결과 없음’을 구분합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 조건부 간선으로 검색 생략

제공 그래프는 판단을 돌리는 자리에 세 낱말을 검사하는 로컬 규칙을 둡니다. model에서 tool로 간 경우에만 rounds가 늘고 다시 model로 돌아옵니다. 실습의 한도는 검색 한 차례입니다. 생성 모델의 실제 도구 선택 성공률을 재는 예제는 아닙니다.


### 셀 01 · 조건부 도구 호출 그래프

decide에서 finish를 돌려주는 두 조건을 찾아보세요. 주차 요금 한 건을 넣습니다.


In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class Loop(TypedDict):
    question: str
    rounds: int
    documents: list[dict]

def decide(state):
    if state["rounds"] >= 1:
        return "finish"
    if any(term in state["question"] for term in terms):
        return "search"
    return "finish"

def run_tool(state):
    return {"documents": retrieve(state["question"], 1), "rounds": state["rounds"] + 1}

builder = StateGraph(Loop)
builder.add_node("model", lambda state: {})
builder.add_node("tool", run_tool)
builder.add_edge(START, "model")
builder.add_conditional_edges("model", decide, {"search": "tool", "finish": END})
builder.add_edge("tool", "model")
agent = builder.compile()

request = {'question':'주차 요금은 얼마인가요?', 'rounds':0, 'documents':[]}
print(agent.invoke(request)['rounds'])

0


0은 검색을 실행하지 않았다는 기록입니다. 검색을 했지만 결과가 빈 경우도 문서 목록은 []이므로 목록만 봐서는 두 상황을 구별하기 어렵습니다.


### decide()가 반환하는 두 경로

<p>model 노드는 빈 변경분을 반환합니다. model 뒤의 조건 함수 decide가 rounds와 질문을 읽어 search 또는 finish를 반환합니다. 등록한 대응표에 따라 search는 tool로, finish는 END로 이어집니다.</p>


### 반복을 결정하는 상태 세 칸

<p>question은 어떤 자료가 필요한지 판단할 입력이고, documents는 방금 찾은 문서입니다. rounds는 tool을 실행할 때마다 1 증가합니다. 판단 함수 자체는 이 값을 변경하지 않습니다.</p>


### 셀 02 · 필수 검색 노드의 실행 경로

이번에는 분기 없이 검색을 거치는 실제 그래프입니다. 한 요청의 검색 횟수와 목록을 함께 읽으세요.


In [2]:
fixed = StateGraph(Loop)
fixed.add_node("search", run_tool)
fixed.add_edge(START, "search")
fixed.add_edge("search", END)
mandatory = fixed.compile()
fixed_result = mandatory.invoke(request)
print(fixed_result["rounds"], fixed_result["documents"])

1 []


1과 빈 목록이 나옵니다. 같은 질문에서 선택 검색은 위 셀처럼 0회, 필수 검색은 1회 호출 뒤 빈 결과였습니다.

검색 횟수 1과 빈 목록 []을 함께 읽으세요. 이 경로는 검색 단계를 반드시 지나지만, 자료에 주차 안내가 새로 생기지는 않습니다. 호출 여부를 점검하는 테스트와 근거가 있는지를 점검하는 테스트가 각각 필요한 사례입니다.


### 같은 주차 질문을 고정 간선에 넣으면

<p>필수 검색 그래프는 START에서 search로 곧장 들어갑니다. 조건 간선이 없으므로 주차 질문도 한 번 검색해 rounds=1, documents=[]가 됩니다. 선택 그래프의 rounds=0과 구별할 수 있습니다.</p>


## 도구 호출 후 재분기와 종료 한도

일단 선택한 도구의 결과가 상태에 들어오면 다시 판단합니다. 완료한 검색을 세지 않거나 종료 조건보다 검색 조건을 먼저 검사하면 같은 요청을 되풀이할 수 있습니다.


### 도구가 돌려준 뒤 같은 model에서 종료 경로를 고릅니다

model → tool → model로 한 바퀴 도는 동안 rounds가 0에서 1로 바뀝니다. 두 번째 decide가 finish를 반환하므로 tool에 다시 들어가지 않습니다.

1. START · question = 대출 기간은 며칠인가요?
2. model · decide의 반환 = search
3. tool · documents[0].id = library-1
4. model · decide의 반환 = finish
5. END · 출력 = 1 library-1


### 셀 03 · 도구 요청과 상태 추적

같은 초기 상태에 대출 질문을 넣어 결과와 검색 횟수를 확인합니다.

이 대출 질문에서 따라갈 길은 model → tool → model → END입니다. 뒤의 참고 그림에는 답변 작성과 호출 상한에 도달했을 때의 길까지 있습니다. 먼저 이 질문에서 실제로 지난 길만 짚어 보세요.


In [3]:
result = agent.invoke({**request, 'question':'대출 기간은 며칠인가요?'})
print(result['rounds'], result['documents'][0]['id'])

1 library-1


1회 검색에서 library-1을 찾았습니다. tool 뒤에 model로 돌아왔지만 rounds가 1이어서 두 번째 검색을 요청하지 않습니다.


### 검색에 쓰인 근거가 결과 상태에 남는 시점

<p>대출 질문에서는 tool이 library-1을 찾습니다. 이 셀의 마지막 출력은 검색 회차와 문서 ID입니다. 이 그래프에는 별도의 답변 생성 노드가 없으므로, 문서를 찾은 시점과 답변 생성 시점을 혼동하지 마세요.</p>

<h3>종료 후 읽을 수 있는 값</h3>

<p>대출 질문의 최종 상태에서 rounds는 1, documents의 첫 id는 library-1입니다. 조건부 간선이 model에서 END로 이어진 뒤에는 추가 검색이나 답변 생성을 실행하지 않습니다.</p>


### tool 다음에 model로 되돌아가는 이유

<p>tool은 documents와 rounds를 함께 갱신합니다. tool → model 간선을 통해 다시 판단할 때에는 같은 질문이어도 rounds가 1입니다. 상한 조건이 먼저 적용되어 finish로 갑니다.</p>


### 검색 생략이 적절했는지 판단하기

<p>규칙은 실제 의도 대신 질문에 대출·휴관·반납이라는 낱말이 있는지만 봅니다. 예를 들어 “책을 빌릴 수 있는 기간?”은 대출 질문이지만 이 조건에서는 검색을 생략합니다. rounds=0만 보고 올바른 응답이라고 결론짓기 어렵습니다.</p>


### 과제 04 · 반복 상한의 종료 분기

대출 질문이라도 이미 한 번 검색했다면 decide가 무엇을 반환하는지 확인하세요.


In [4]:
# 결과 상태 result를 판단 함수에 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(decide(result))</code></pre>
<p>실행 결과</p><pre><code>finish</code></pre>
<p>finish가 돌아옵니다. 필수 검색이 서비스 규칙이면 누락을 막기 위해 고정 노드를 두고, 선택 검색을 쓴다면 생략과 상한 종료를 따로 기록하세요.</p>
</details>


### 종료 조건 rounds >= 1

<p>첫 검색 후 결과 상태를 decide에 그대로 넣어 보세요. 질문에 대출이 있어도 rounds &gt;= 1 검사가 먼저 실행됩니다. 이 순서 덕분에 같은 검색으로 되돌아가지 않습니다.</p>


## 돌아보기

검색이 반드시 필요한 팀 서비스인지 먼저 합의하세요. 한 번 더 판단할 때의 비용과 검색 누락을 확인할 사례를 함께 정해야 구조를 고를 수 있습니다.

### 참고 구현의 정상 왕복과 마지막 answer 노드

**검색 요청을 만든다**　아직 R1 검색은 실행되지 않았습니다.

```
decision_calls=1
retrieval_calls=0
generation_calls=0
```

**검색한 문서를 받는다**　ToolMessage를 붙이고 문서 목록을 갱신합니다.

```
tool_rounds=1
retrieval_calls=1
documents=[library-1]
```

**추가 요청 없이 마친다**　마지막 AIMessage의 tool_calls는 빈 목록입니다.

```
추가 검색 요청 없음 → answer
상한 뒤 또 요청 → 추가 검색 없이 중단
```

**같은 A1로 인용을 확인한다**　고정 생성기는 한 번 실행됩니다.

```
generation_calls=1
status=answered
sources=[library-1]
```
